In [72]:
# Data handling
import numpy as np
import pandas as pd

# Preprocessing
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer


# Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Model selection
from sklearn.model_selection import train_test_split, GridSearchCV

# Pipeline
from sklearn.pipeline import Pipeline


### Encode categorical data

In [63]:
df_train = pd.read_csv("trainCA4DAT200.csv")

# categorize nominal features
df_train["InternationalPlan"] = df_train["InternationalPlan"].map({"yes": 1, "no": 0})
df_train["VoiceMailPlan"] = df_train["VoiceMailPlan"].map({"yes": 1, "no": 0})

# categorical features
state_ohe = OneHotEncoder(drop='first', sparse_output=False)
out = state_ohe.fit_transform(df_train[['State']]) # expects a 2D array
# turn into data frame
df_ohe = pd.DataFrame(out, columns=state_ohe.get_feature_names_out())

# combine back into the original data frame
df_train = pd.concat([df_train.drop(['State'], axis=1), df_ohe], axis=1)

df_train.head()



,CustomerId,AccountLength,AreaCode,InternationalPlan,VoiceMailPlan,NumVoiceMailMessages,DayMinutes,DayCalls,DayCharge,EveMinutes,...,State_SD,State_TN,State_TX,State_UT,State_VA,State_VT,State_WA,State_WI,State_WV,State_WY
0,952920,87,510,0,0,0,223.2,109,37.94,127.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,895150,114,510,0,1,28,225.8,94,38.39,193.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,636987,55,415,0,0,0,180.1,77,30.62,235.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,538188,100,510,1,0,0,68.5,110,11.65,337.1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,947531,74,408,0,1,46,277.7,109,47.21,270.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Split data and expermient with gridsearch

In [64]:

x = df_train.drop(columns=["Churn", "CustomerId"])
y = df_train["Churn"]

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.4, random_state=42, stratify=y)

clf = RandomForestClassifier()

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10],
    'min_samples_split': [2, 5, 10],
}

grid = GridSearchCV(
    clf, 
    param_grid=param_grid, 
    cv=5
)
grid.fit(X_train, y_train)



,estimator,RandomForestClassifier()
,param_grid,"{'max_depth': [None, 10], 'min_samples_split': [2, 5, ...], 'n_estimators': [50, 100, ...]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


In [84]:

best_model = grid.best_estimator_
best_model.fit(X_train, y_train)
print(best_model.score(X_test, y_test))

0.8642857142857143


### Messing around with pipelines combined with grid search

In [83]:
# Try with pipeline for scaling and logistic regression
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())
])

param_grid = {
    'classifier__C': [0.1, 1, 10],
    'classifier__penalty': ['l1', 'l2'],
    'classifier__solver': ['liblinear']
}

grid = GridSearchCV(
    pipeline, 
    param_grid=param_grid, 
    cv=5,
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_
best_model.fit(X_train, y_train)
best_model.score(X_test, y_test)

0.8642857142857143

In [81]:
# Try with pipeline for KNN
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', KNeighborsClassifier())
])


param_grid = {
    'classifier__n_neighbors': [3, 5, 7],
    'classifier__weights': ['uniform', 'distance'],
    'classifier__metric': ['euclidean', 'manhattan']
}

grid = GridSearchCV(
    pipeline, 
    param_grid=param_grid, 
    cv=5,
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_
best_model.fit(X_train, y_train)
best_model.score(X_test, y_test)

0.8592857142857143

In [82]:
# Try with pipeline for Randomforest
pipeline = Pipeline([
    ('classifier', RandomForestClassifier())
])


param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 10],
    'classifier__min_samples_split': [2, 5, 10],
}

grid = GridSearchCV(
    pipeline, 
    param_grid=param_grid, 
    cv=5,
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_
best_model.fit(X_train, y_train)
best_model.score(X_test, y_test)

0.9378571428571428